# What is RAG

1. source: https://www.youtube.com/watch?v=JktYwBIDErk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=5
2. for RAG, we need (Search, Prompt, and LLM)
3. main idea:

    3.1 given a query (question), we do search text using minsearch, sqlitesearch, ..... from the documents and we get search_result, then we build context (clean the search_result),
    
    3.2 we build a prompt using the query and the context,
    
    3.3 we build our LLM and we pass the prompt as input,
    
    3.4 we parse the result to the response

# Build llm that answer only a question depending on its training data ONLY

Note: we can also build functions that does chain

In [ ]:
# this is without using chain
from dotenv import load_dotenv
import os
load_dotenv()

"""
    def llm(prompt):
    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=prompt
    )
    return response.output_text

"""
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
import textwrap


def llm(prompt):
    llm_model = ChatOllama(
            model="gemma3:4b",
            base_url="http://localhost:11434",
            temperature = 0,
        )
    response = llm_model.invoke(prompt)
    response_clean = textwrap.fill(
            response.content, 
            # width=100
        )
    return(response_clean )

# the following is better, since it uses chain

def llm_with_chain( prompt):
    input_prompt = ChatPromptTemplate.from_template("You are the AI agent, explain the {content}")
    llm_model = ChatOllama(
                model="gemma3:4b",
                base_url="http://localhost:11434",
                temperature = 0,
            )
        
    output_parser = StrOutputParser()

    chain = input_prompt | llm_model | output_parser | RunnableLambda(lambda x: x.replace("\n","").strip() )
    response = chain.invoke({"content": prompt})
    clean_response = textwrap.fill(
            response, 
            # width=100
        )
    return(clean_response)




question= "I just discovered the course. Can I join now?"

print(f"Anwer using llm model ONLY: {llm(question)}")
print(f"Anwer using LLM with framework chain:{llm_with_chain( question)}")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Anwer using llm model ONLY: Please tell me which course you're referring to! 😊 I need to know the
name of the course or where you found it to let you know if you can
join and what the requirements might be.   For example, you could say:
* "I just discovered the 'Introduction to Python Programming' course
on Coursera..." * "I just saw the 'Digital Marketing Fundamentals'
course on Skillshare..."  Once you tell me which course it is, I can
help you figure out if you can join and what you need to do.
Anwer using LLM with framework chain:Okay, fantastic that you've found the course! Let’s figure out if you
can join right away. To answer your question – “Can I join now?” – I
need a little more information.**Here’s what we need to determine if
you can enroll:**1. **Which Course Are You Referring To?**  I don't
have context about *which* course you discovered. Please tell me:   *
**What is the name of the course?** (e.g., "Introduction to Python
Programming," "Digital Marketing Fundamentals,"

# Answer/Question depending on context (document)

In [ ]:
# we can use ChatPromtTemplate.from_template OR ChatPromptTemplate.from_messages
# 1. With ChatPromptTemplate.from_template

from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


docs = """General Course-Related Questions
#I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

 """

# Using ChatPromptTemplate.from_template
# prompt = ChatPromptTemplate.from_template(
#     """You are a strict, helpful assistant. Answer the user's question based ONLY on the provided context. 
# If the answer cannot be found in the context, say 'I cannot find the answer in the text.' 
# Do not use external knowledge or make up facts.

# Context:
# {context}

# Question: {question}"""
# )

prompt = ChatPromptTemplate.from_messages([
    (
        "system", 
        "You are a strict, helpful assistant. Answer the user's question based ONLY on the provided context. "
        "If the answer cannot be found in the context, say 'I cannot find the answer in the text.' "
        "Do not use external knowledge or make up facts.\n\n"
        "Context:\n{context}"
    ),
    ("human", "{question}")
])

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.9,
    base_url="http://localhost:11434",
)

chain = prompt | llm | StrOutputParser()
chain.invoke({  "context": docs,
     "question":  "I just discovered the course. Can I join now?"
})

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

# using create_stuff_document

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.documents import Document

docs = """General Course-Related Questions
#I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

 """
docs_list = [Document(page_content=docs)]

prompt = ChatPromptTemplate.from_messages([
    (
        "system", 
        "You are a strict, helpful assistant. Answer the user's question based ONLY on the provided context. "
        "If the answer cannot be found in the context, say 'I cannot find the answer in the text.' "
        "Do not use external knowledge or make up facts.\n\n"
        "Context:\n{context}"
    ),
    ("human", "{question}")
])

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.9,
    base_url="http://localhost:11434",
)

documents_chain = create_stuff_documents_chain(llm, prompt)
documents_chain.invoke({  "context": docs_list,
     "question":  "I just discovered the course. Can I join now?"
})

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'